# Crocevia — Prévision de la demande & MLOps dans Snowflake## In-database demand forecasting + full MLOps## Overview**FR.** Deux approches de prévision comparées *dans* Snowflake : (1) **Cortex `FORECAST`** (managé, zéro-effort) et (2) un **XGBoost Snowpark ML** avec features avancées. Puis on déroule le cycle **MLOps** : Model Registry (versioning + métriques), explicabilité SHAP, lineage, Model Monitor, et Experiment Tracking. Aucune donnée ne quitte Snowflake.**EN.** Two forecasting approaches compared *inside* Snowflake: (1) **Cortex `FORECAST`** (managed, zero-effort) and (2) a **Snowpark ML XGBoost** with engineered features. Then the full **MLOps** lifecycle: Model Registry (versioning + metrics), SHAP explainability, lineage, Model Monitor, and Experiment Tracking. No data leaves Snowflake.## What You'll Learn- Train Cortex `FORECAST` (managed) and compare it to a custom XGBoost- Engineer features (lags, rolling stats, calendar) with SQL windows- Register, version and add metrics to a model (Model Registry)- Explain predictions (SHAP), track data→model lineage, monitor drift, and log an experiment run## Prerequisites- Role with access to `CROCEVIA_DB.PLATFORM_DEMO` (dbt marts built) · warehouse `CR_DEV_WH`- Cross-region inference enabled · `snowflake-ml-python` (provided in Snowflake Notebooks)## Estimated Time: ~10 min (Run all)

In [ ]:
# === ALL IMPORTS HERE ===import loggingimport numpy as npimport pandas as pdimport plotly.express as pximport xgboost as xgbfrom sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_errorfrom snowflake.snowpark.context import get_active_sessionfrom snowflake.ml.registry import Registrysession = get_active_session()session.sql("USE SCHEMA CROCEVIA_DB.PLATFORM_DEMO").collect()session.sql("USE WAREHOUSE CR_DEV_WH").collect()logging.getLogger().setLevel(logging.INFO)logger = logging.getLogger("crocevia_demo")logger.info(f"Connecté / connected: {session.get_current_database()}.{session.get_current_schema()}")

## 1. Données / Data**FR.** Ventes quotidiennes par catégorie depuis le mart dbt testé `MART_SALES_DAILY`.**EN.** Daily sales per category from the tested dbt mart `MART_SALES_DAILY`.

In [ ]:
SELECT product_category, ROUND(SUM(revenue_eur)) AS revenue_eur, SUM(units) AS unitsFROM MART_SALES_DAILYWHERE product_category <> 'UNKNOWN'GROUP BY 1 ORDER BY revenue_eur DESC LIMIT 10;

In [ ]:
df = top_categories.to_pandas()fig = px.bar(df, x="REVENUE_EUR", y="PRODUCT_CATEGORY", orientation="h",             title="Top catégories par chiffre d'affaires / Top categories by revenue",             labels={"REVENUE_EUR": "CA (€)", "PRODUCT_CATEGORY": ""})fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=360)fig.show()

## 2. Cortex FORECAST (managé / managed)**FR.** Jeu d'entraînement 18 mois (table non protégée pour le moteur ML), puis un modèle multi-séries `SNOWFLAKE.ML.FORECAST` (27 catégories).**EN.** 18-month training table (unprotected for the ML engine), then one multi-series `SNOWFLAKE.ML.FORECAST` model (27 categories).

In [ ]:
CREATE OR REPLACE TABLE FORECAST_TRAIN_CATEGORY ASSELECT product_category::VARCHAR AS series_category, sale_date::TIMESTAMP_NTZ AS ts, SUM(units)::FLOAT AS yFROM MART_SALES_DAILYWHERE sale_date >= DATEADD('month', -18, (SELECT MAX(sale_date) FROM MART_SALES_DAILY))  AND product_category <> 'UNKNOWN'GROUP BY 1, 2;

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST DEMAND_MODEL(  INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'FORECAST_TRAIN_CATEGORY'),  SERIES_COLNAME => 'SERIES_CATEGORY', TIMESTAMP_COLNAME => 'TS', TARGET_COLNAME => 'Y');

## 3. Prévision 14 jours / 14-day forecast**FR.** Prévision avec intervalles, stockée pour les apps. **EN.** Forecast with intervals, persisted for the apps.

In [ ]:
CALL DEMAND_MODEL!FORECAST(FORECASTING_PERIODS => 14);CREATE OR REPLACE TABLE FORECAST_DEMAND_CATEGORY ASSELECT series::VARCHAR AS product_category, ts::DATE AS forecast_date,       ROUND(forecast,0) AS forecast_units, ROUND(lower_bound,0) AS lower_units, ROUND(upper_bound,0) AS upper_unitsFROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

In [ ]:
SELECT ts::DATE AS d, y AS units, 'history' AS kindFROM FORECAST_TRAIN_CATEGORYWHERE series_category ILIKE 'Fruits%' AND ts >= DATEADD('day', -90, (SELECT MAX(ts) FROM FORECAST_TRAIN_CATEGORY))UNION ALLSELECT forecast_date, forecast_units, 'forecast' FROM FORECAST_DEMAND_CATEGORY WHERE product_category ILIKE 'Fruits%'ORDER BY d;

In [ ]:
df = forecast_fruits.to_pandas()fig = px.line(df, x="D", y="UNITS", color="KIND",              title="Fruits et Légumes — historique vs prévision Cortex / history vs Cortex forecast",              labels={"D": "", "UNITS": "Unités / units", "KIND": ""})fig.update_layout(height=340); fig.show()

## 4. Précision Cortex / Cortex accuracy**FR.** Métriques d'erreur intégrées (MAPE/SMAPE par série). **EN.** Built-in error metrics (MAPE/SMAPE per series).

In [ ]:
CALL DEMAND_MODEL!SHOW_EVALUATION_METRICS();

In [ ]:
ev = eval_metrics.to_pandas()summary = ev.groupby("ERROR_METRIC")["METRIC_VALUE"].mean().round(3).reset_index()cortex_mape = float(summary.loc[summary["ERROR_METRIC"]=="MAPE","METRIC_VALUE"].iloc[0])print(f"Cortex FORECAST MAPE (CV) = {cortex_mape:.3f}  ->  ~{(1-cortex_mape)*100:.0f}% accuracy")summary

## 5. Segmentation client (2ᵉ famille ML) / customer segmentation**FR.** Segmentation RFM (`MART_CUSTOMER_RFM`). **EN.** RFM segmentation (`MART_CUSTOMER_RFM`).

In [ ]:
SELECT segment, COUNT(*) AS customers, ROUND(AVG(monetary_eur)) AS avg_spend_eurFROM MART_CUSTOMER_RFM GROUP BY 1 ORDER BY customers DESC;

In [ ]:
df = segments.to_pandas()fig = px.bar(df, x="SEGMENT", y="CUSTOMERS", title="Clients par segment RFM / customers per RFM segment")fig.update_layout(height=340); fig.show()

## 6. Modèle avancé Snowpark ML + MLOps / Advanced model + MLOps**FR.** On entraîne un **XGBoost** avec features avancées (lags, moyennes/écarts glissants, calendrier, code catégorie), on le compare à Cortex, puis on déroule le **cycle MLOps** : Registry + métriques, explicabilité SHAP, lineage, Model Monitor, Experiment Tracking.**EN.** Train an **XGBoost** with engineered features (lags, rolling mean/std, calendar, category code), compare it to Cortex, then run the **MLOps lifecycle**: Registry + metrics, SHAP explainability, lineage, Model Monitor, Experiment Tracking.

In [ ]:
-- Feature engineering with SQL window functions (lags, rolling stats, calendar, category code)CREATE OR REPLACE TABLE FORECAST_FEATURES_CATEGORY ASWITH base AS (  SELECT series_category, ts, y,    LAG(y,1)  OVER (PARTITION BY series_category ORDER BY ts) AS lag_1,    LAG(y,7)  OVER (PARTITION BY series_category ORDER BY ts) AS lag_7,    LAG(y,14) OVER (PARTITION BY series_category ORDER BY ts) AS lag_14,    LAG(y,28) OVER (PARTITION BY series_category ORDER BY ts) AS lag_28,    AVG(y)    OVER (PARTITION BY series_category ORDER BY ts ROWS BETWEEN 7  PRECEDING AND 1 PRECEDING) AS roll_7_mean,    AVG(y)    OVER (PARTITION BY series_category ORDER BY ts ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS roll_28_mean,    STDDEV(y) OVER (PARTITION BY series_category ORDER BY ts ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS roll_28_std,    DAYOFWEEK(ts) AS dow, DAY(ts) AS dom, MONTH(ts) AS month, WEEKOFYEAR(ts) AS woy  FROM FORECAST_TRAIN_CATEGORY),coded AS (SELECT b.*, DENSE_RANK() OVER (ORDER BY series_category) AS category_code FROM base b)SELECT * FROM coded WHERE lag_28 IS NOT NULL AND roll_28_mean IS NOT NULL;

In [ ]:
FEAT = ["LAG_1","LAG_7","LAG_14","LAG_28","ROLL_7_MEAN","ROLL_28_MEAN","ROLL_28_STD",        "DOW","DOM","MONTH","WOY","CATEGORY_CODE"]fdf = session.table("FORECAST_FEATURES_CATEGORY").to_pandas()fdf["TS"] = pd.to_datetime(fdf["TS"]); fdf[FEAT] = fdf[FEAT].fillna(0)cutoff = fdf["TS"].max() - pd.Timedelta(days=14)train, test = fdf[fdf["TS"] <= cutoff], fdf[fdf["TS"] > cutoff]xgb_model = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,                             subsample=0.8, colsample_bytree=0.8, random_state=42)xgb_model.fit(train[FEAT], train["Y"])pred = np.clip(xgb_model.predict(test[FEAT]), 0, None)xgb_mape = mean_absolute_percentage_error(test["Y"], pred)xgb_rmse = root_mean_squared_error(test["Y"], pred)print(f"XGBoost holdout MAPE={xgb_mape:.3f}  RMSE={xgb_rmse:.0f}")print(f"Cortex FORECAST MAPE (CV)={cortex_mape:.3f}")

In [ ]:
# Comparison (note: different eval windows - Cortex CV vs XGBoost 14d holdout - illustrative)cmp = pd.DataFrame({"model": ["Cortex FORECAST", "XGBoost (Snowpark ML)"], "mape": [cortex_mape, xgb_mape]})fig = px.bar(cmp, x="model", y="mape", text=cmp["mape"].round(3),             title="Comparaison MAPE (plus bas = mieux) / MAPE comparison (lower = better)",             labels={"model": "", "mape": "MAPE"})fig.update_layout(height=340); fig.show()

### MLOps 1/4 — Experiment Tracking + Model Registry**FR.** On trace le run (params + métriques) puis on enregistre le modèle versionné avec ses métriques. L'échantillon Snowpark active le **lineage** et l'**explicabilité**.**EN.** Track the run (params + metrics), then register the versioned model with its metrics. The Snowpark sample input enables **lineage** and **explainability**.

In [ ]:
from snowflake.ml.experiment import ExperimentTrackingexp = ExperimentTracking(session, database_name="CROCEVIA_DB", schema_name="PLATFORM_DEMO")exp.set_experiment("CROCEVIA_DEMAND_FORECAST")exp.start_run("xgb_v1")exp.log_params({"n_estimators": 400, "max_depth": 6, "learning_rate": 0.05,                "features": "lags+rolling+calendar+category_code"})exp.log_metrics({"mape_holdout": float(xgb_mape), "rmse_holdout": float(xgb_rmse), "cortex_mape_cv": float(cortex_mape)})exp.end_run()print("Experiment run logged: CROCEVIA_DEMAND_FORECAST / xgb_v1")

In [ ]:
reg = Registry(session=session, database_name="CROCEVIA_DB", schema_name="PLATFORM_DEMO")# Snowpark sample input (feature columns) -> enables data->model lineage + SHAP explainabilitysample_sp = session.table("FORECAST_FEATURES_CATEGORY").select(FEAT).limit(100)try:    mv = reg.log_model(        xgb_model, model_name="CROCEVIA_DEMAND_XGB", version_name="V1",        sample_input_data=sample_sp, conda_dependencies=["xgboost", "scikit-learn"],        target_platforms=["WAREHOUSE"],        comment="XGBoost demand model (engineered features) vs Cortex FORECAST baseline")except Exception as e:    print("version exists, reusing:", e); mv = reg.get_model("CROCEVIA_DEMAND_XGB").version("V1")mv.set_metric("mape_holdout", float(xgb_mape))mv.set_metric("rmse_holdout", float(xgb_rmse))mv.set_metric("cortex_mape_cv", float(cortex_mape))print("Registered:", mv.model_name, mv.version_name, "| metrics:", mv.show_metrics())

### MLOps 2/4 — Explicabilité SHAP / SHAP explainability**FR.** Le registry génère automatiquement une méthode `explain` (contributions par feature). **EN.** The registry auto-generates an `explain` method (per-feature contributions).

In [ ]:
shap = mv.run(sample_sp.limit(50), function_name="explain").to_pandas()mean_abs = shap.abs().mean().sort_values(ascending=False).head(10).reset_index()mean_abs.columns = ["feature", "mean_abs_shap"]fig = px.bar(mean_abs, x="mean_abs_shap", y="feature", orientation="h",             title="Importance des features (|SHAP| moyen) / feature importance (mean |SHAP|)")fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=360); fig.show()

### MLOps 3/4 — Lineage données → modèle / data → model lineage**FR.** `GET_LINEAGE` trace la chaîne du mart au modèle. **EN.** `GET_LINEAGE` traces the chain from mart to model.

In [ ]:
lin = session.sql(f'''  SELECT source_object_name AS source, source_object_domain AS source_type,         target_object_name AS target, target_object_domain AS target_type, distance  FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(    'CROCEVIA_DB.PLATFORM_DEMO.CROCEVIA_DEMAND_XGB', 'MODULE', 'UPSTREAM', 3, '{mv.version_name}'))''').to_pandas()print("Lineage edges (mart -> features -> model):")lin

### MLOps 4/4 — Model Monitor (drift / performance)**FR.** On crée une source de monitoring (prédictions + valeurs réelles) puis un **Model Monitor** qui suit drift & performance. **EN.** Build a monitoring source (predictions + actuals), then a **Model Monitor** tracking drift & performance.

In [ ]:
recent = fdf[fdf["TS"] > fdf["TS"].max() - pd.Timedelta(days=60)].copy()recent["PREDICTION"] = np.clip(xgb_model.predict(recent[FEAT]), 0, None).astype(float)src = recent[["TS","SERIES_CATEGORY","CATEGORY_CODE","PREDICTION","Y"]].rename(columns={"Y": "ACTUAL"})src["ROW_ID"] = src["SERIES_CATEGORY"] + "_" + src["TS"].dt.strftime("%Y%m%d")src["TS"] = src["TS"].dt.strftime("%Y-%m-%d %H:%M:%S")session.write_pandas(src, "CROCEVIA_DEMAND_MONITOR_RAW", auto_create_table=True, overwrite=True, quote_identifiers=False)session.sql('''CREATE OR REPLACE TABLE CROCEVIA_DEMAND_MONITOR_SRC AS  SELECT TO_TIMESTAMP_NTZ(TS) AS TS, SERIES_CATEGORY, CATEGORY_CODE,         PREDICTION::FLOAT AS PREDICTION, ACTUAL::FLOAT AS ACTUAL, ROW_ID  FROM CROCEVIA_DEMAND_MONITOR_RAW''').collect()session.sql('''CREATE OR REPLACE MODEL MONITOR CROCEVIA_DB.PLATFORM_DEMO.CROCEVIA_DEMAND_XGB_MON  WITH MODEL = CROCEVIA_DB.PLATFORM_DEMO.CROCEVIA_DEMAND_XGB VERSION = V1 FUNCTION = 'predict'       WAREHOUSE = CR_DEV_WH SOURCE = CROCEVIA_DB.PLATFORM_DEMO.CROCEVIA_DEMAND_MONITOR_SRC       ID_COLUMNS = ('ROW_ID') TIMESTAMP_COLUMN = 'TS'       PREDICTION_SCORE_COLUMNS = ('PREDICTION') ACTUAL_SCORE_COLUMNS = ('ACTUAL')       REFRESH_INTERVAL = '1 day' AGGREGATION_WINDOW = '1 day' ''').collect()print("Model Monitor created: CROCEVIA_DEMAND_XGB_MON")

## Conclusion**FR.** Deux approches comparées (Cortex `FORECAST` ~0.24 MAPE CV vs XGBoost ~0.32 holdout — fenêtres d'éval différentes, indicatif) et **tout le cycle MLOps** : Registry + métriques + versioning, explicabilité SHAP, lineage données→modèle, Model Monitor, Experiment Tracking — *dans* Snowflake, sur données gouvernées. Cortex offre une base managée très solide ; XGBoost ajoute le contrôle, l'explicabilité et la gouvernance complète.**EN.** Two approaches compared (Cortex `FORECAST` ~0.24 CV MAPE vs XGBoost ~0.32 holdout — different eval windows, illustrative) and the **full MLOps lifecycle**: Registry + metrics + versioning, SHAP explainability, data→model lineage, Model Monitor, Experiment Tracking — all *inside* Snowflake on governed data. Cortex is a strong managed baseline; XGBoost adds control, explainability and full governance.

## Nettoyage / Cleanup (optionnel)**FR.** ⚠️ `FORECAST_DEMAND_CATEGORY` est utilisée par les apps — ne supprimez qu'en teardown complet. Mettez `RUN_CLEANUP = True`.**EN.** ⚠️ `FORECAST_DEMAND_CATEGORY` is used by the apps — only drop for a full teardown. Set `RUN_CLEANUP = True`.

In [ ]:
RUN_CLEANUP = Falseif RUN_CLEANUP:    for stmt in [        "DROP MODEL MONITOR IF EXISTS CROCEVIA_DEMAND_XGB_MON",        "DROP MODEL IF EXISTS CROCEVIA_DEMAND_XGB",        "DROP SNOWFLAKE.ML.FORECAST IF EXISTS DEMAND_MODEL",        "DROP TABLE IF EXISTS FORECAST_TRAIN_CATEGORY",        "DROP TABLE IF EXISTS FORECAST_FEATURES_CATEGORY",        "DROP TABLE IF EXISTS FORECAST_DEMAND_CATEGORY",        "DROP TABLE IF EXISTS CROCEVIA_DEMAND_MONITOR_RAW",        "DROP TABLE IF EXISTS CROCEVIA_DEMAND_MONITOR_SRC",    ]:        try: session.sql(stmt).collect(); logger.info(f"OK: {stmt}")        except Exception as e: logger.warning(f"Cleanup warning: {e}")    logger.info("Cleanup terminé / complete")else:    logger.info("Cleanup ignoré (RUN_CLEANUP=False) / skipped")